In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler

# Fixed seed so the split, oversampling and network initialisation are reproducible
SEED = 42

# QUICK_RUN=1 shrinks the neural-network grid search to one short run (used by CI as a smoke test)
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"

### Dataset:
Dua, D. and Graff, C. (2019). UCI Machine Learning Repository [http://archive.ics.uci.edu/ml]. Irvine, CA: University of California, School of Information and Computer Science.

Donated by:
P. Savicky
Institute of Computer Science, AS of CR
Czech Republic
savicky '@' cs.cas.cz

In [ ]:
cols = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
df = pd.read_csv("magic04.data", names=cols)
df.head()

In [ ]:
df["class"] = (df["class"] == "g").astype(int)

In [ ]:
df.head()

In [ ]:
for label in cols[:-1]:
  plt.hist(df[df["class"]==1][label], color='blue', label='gamma', alpha=0.7, density=True)
  plt.hist(df[df["class"]==0][label], color='red', label='hadron', alpha=0.7, density=True)
  plt.title(label)
  plt.ylabel("Probability")
  plt.xlabel(label)
  plt.legend()
  plt.show()

#Train, validation, test datasets

In [ ]:
# Shuffle once, then slice 60% / 20% / 20% (iloc keeps each split a DataFrame)
shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_train, n_valid = int(0.6 * len(shuffled)), int(0.8 * len(shuffled))
train, valid, test = shuffled.iloc[:n_train], shuffled.iloc[n_train:n_valid], shuffled.iloc[n_valid:]

In [ ]:
def scale_dataset(dataframe, scaler, oversample=False):
  X = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  X = scaler.transform(X)

  if oversample:
    ros = RandomOverSampler(random_state=SEED)
    X, y = ros.fit_resample(X, y)

  data = np.hstack((X, np.reshape(y, (-1, 1))))

  return data, X, y

In [ ]:
# Fit the scaler on the training split only, then apply the same transform to validation and test
scaler = StandardScaler().fit(train[train.columns[:-1]].values)

train, X_train, y_train = scale_dataset(train, scaler, oversample=True)
valid, X_valid, y_valid = scale_dataset(valid, scaler, oversample=False)
test, X_test, y_test = scale_dataset(test, scaler, oversample=False)

# kNN

In [ ]:
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

In [ ]:
y_pred = knn_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
nb_model = GaussianNB()
nb_model = nb_model.fit(X_train, y_train)

In [ ]:
y_pred = nb_model.predict(X_test)
print(classification_report(y_test, y_pred))

# Log Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lg_model = LogisticRegression()
lg_model = lg_model.fit(X_train, y_train)

In [ ]:
y_pred = lg_model.predict(X_test)
print(classification_report(y_test, y_pred))

# SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
svm_model = SVC()
svm_model = svm_model.fit(X_train, y_train)

In [ ]:
y_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_pred))

# Neural Net

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

In [ ]:
def plot_history(history):
  _, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
  ax1.plot(history.history['loss'], label='loss')
  ax1.plot(history.history['val_loss'], label='val_loss')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Binary crossentropy')
  ax1.legend()
  ax1.grid(True)

  ax2.plot(history.history['accuracy'], label='accuracy')
  ax2.plot(history.history['val_accuracy'], label='val_accuracy')
  ax2.set_xlabel('Epoch')
  ax2.set_ylabel('Accuracy')
  ax2.legend()
  ax2.grid(True)

  plt.show()

In [ ]:
def train_model(X_train, y_train, X_valid, y_valid, num_nodes, dropout_prob, lr, batch_size, epochs):
  nn_model = tf.keras.Sequential([
      tf.keras.Input(shape=(X_train.shape[1],)),
      tf.keras.layers.Dense(num_nodes, activation='relu'),
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(num_nodes, activation='relu'),
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(1, activation='sigmoid')
  ])

  nn_model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy',
                  metrics=['accuracy'])
  # Validate on the held-out validation split. validation_split=0.2 would take the last 20% of the
  # oversampled training array, which RandomOverSampler fills with duplicated hadron rows.
  history = nn_model.fit(
    X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_valid, y_valid), verbose=0
  )

  return nn_model, history

In [ ]:
least_val_loss = float('inf')
least_loss_model = None
if QUICK_RUN:
  epochs, grid_nodes, grid_dropout, grid_lr, grid_batch = 2, [16], [0], [0.001], [128]
else:
  epochs, grid_nodes, grid_dropout, grid_lr, grid_batch = 100, [16, 32, 64], [0, 0.2], [0.01, 0.005, 0.001], [32, 64, 128]
best_params = None
for num_nodes in grid_nodes:
  for dropout_prob in grid_dropout:
    for lr in grid_lr:
      for batch_size in grid_batch:
        print(f"{num_nodes} nodes, dropout {dropout_prob}, lr {lr}, batch size {batch_size}")
        model, history = train_model(X_train, y_train, X_valid, y_valid, num_nodes, dropout_prob, lr, batch_size, epochs)
        plot_history(history)
        val_loss = model.evaluate(X_valid, y_valid, verbose=0)[0]
        if val_loss < least_val_loss:
          least_val_loss = val_loss
          least_loss_model = model
          best_params = (num_nodes, dropout_prob, lr, batch_size)

print(f"Best: {best_params[0]} nodes, dropout {best_params[1]}, lr {best_params[2]}, batch size {best_params[3]}, val loss {least_val_loss:.4f}")

In [ ]:
nn_scores = least_loss_model.predict(X_test, verbose=0).reshape(-1,)
y_pred = (nn_scores > 0.5).astype(int)

In [ ]:
print(classification_report(y_test, y_pred))

# ROC comparison

`magic04.names` notes that accuracy is not a meaningful measure here: classifying a hadron (background) as a gamma is worse than the reverse. It recommends ROC curves and the signal acceptance (gamma true-positive rate) at low hadron false-positive rates. Here each model's continuous score on the test split is compared.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

test_scores = {
  "kNN (k=5)": knn_model.predict_proba(X_test)[:, 1],
  "Naive Bayes": nb_model.predict_proba(X_test)[:, 1],
  "Logistic Regression": lg_model.predict_proba(X_test)[:, 1],
  "SVM": svm_model.decision_function(X_test),
  "Neural network": nn_scores,
}

def tpr_at_fpr(y_true, scores, max_fpr):
  fpr, tpr, _ = roc_curve(y_true, scores)
  return tpr[fpr <= max_fpr].max()

rows = []
plt.figure(figsize=(6, 5))
for name, scores in test_scores.items():
  fpr, tpr, _ = roc_curve(y_test, scores)
  auc = roc_auc_score(y_test, scores)
  plt.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})")
  rows.append({"model": name, "ROC AUC": auc,
               "gamma TPR @ 1% hadron FPR": tpr_at_fpr(y_test, scores, 0.01),
               "gamma TPR @ 5% hadron FPR": tpr_at_fpr(y_test, scores, 0.05)})
plt.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1)
plt.xlabel("Hadron false-positive rate")
plt.ylabel("Gamma true-positive rate")
plt.title("ROC curves on the test split")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print(pd.DataFrame(rows).set_index("model").round(3).to_string())